In [5]:
# Stanley Uche Godfrey
# ustan.godfrey@gmail.com
# A Movie Chatbot, The goal is to build a chatbot 
# that understands natural language queries 
# and retrieves relevant movie information from an IMDb dataset.


In [1]:


# Importing the CSV module for reading and writing CSV files.
import csv

# Importing numpy for numerical operations and handling arrays efficiently.
import numpy as np

# Importing pandas for data manipulation and analysis.
import pandas as pd

# Importing os to interact with the operating system, such as environment variables and file paths.
import os

# Importing getpass to securely handle user input (e.g., API keys or passwords).
import getpass

import math

# Importing the OpenAI library to interact with OpenAI's API services.
from openai import OpenAI

# Import basic libraries
from dotenv import load_dotenv
import pandas as pd






In [2]:
# Store your OpenAI API key

dotenv_dir='../chat_bot/prod_small/' #Replace with your path
print("dotenv_dir",dotenv_dir)  # Debugging statement to check the path to the .env file
env_path=os.path.join(dotenv_dir,'.env')
# Load environment variables from .env file
load_dotenv(env_path)
OPENROUTER_API_KEY = os.getenv('OPEN_ROUTER_KEY')

os.environ["no_proxy"] = "localhost,127.0.0.1,::1"


os.environ["OPEN_ROUTER_KEY"] = OPENROUTER_API_KEY

print(OPENROUTER_API_KEY[0:70]+'...')


dotenv_dir ../chat_bot/prod_small/
sk-or-v1-ec8221e51a89098dc915489ca6820a7b269e84d5d580de0e22a5c28c3a751...


In [3]:
# Load the data
imdb_data = pd.read_csv('IMDb_Dataset.csv')


In [4]:
# View & Understand the data
print(imdb_data.columns.tolist())  # Print column names to understand the structure of the dataset


['Title', 'IMDb Rating', 'Year', 'Certificates', 'Genre', 'Director', 'Star Cast', 'MetaScore', 'Poster-src', 'Duration (minutes)']


In [5]:
# Create movie description for each movie from the details provided in the dataset
movie_description = []
movie_description_len = []
for index, row in imdb_data.iterrows():
    description = f"{row['Title']} is a {row['Genre']} movie directed by {row['Director']}. It stars {row['Star Cast']} and was released in {row['Year']}."
    movie_description.append(description)
    movie_description_len.append(len(description))
imdb_data['description'] = movie_description


In [6]:
# Create a vector store using the created chunks and the embeddings model

# Importing RecursiveCharacterTextSplitter from LangChain for chunking large text into smaller, manageable pieces.
# This helps in optimizing text for processing and retrieval.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Importing OpenAIEmbeddings from LangChain to generate numerical vector representations (embeddings) of text.
# These embeddings capture the semantic meaning of the text for efficient similarity searches.
from langchain_openai import OpenAIEmbeddings

# Importing FAISS (Facebook AI Similarity Search) from LangChain's community package.
# FAISS is used for storing and retrieving embeddings efficiently by finding similar vectors.
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores import InMemoryVectorStore
# Split the input text using Recursive Character Chunking
# See this for more details https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/recursive_text_splitter/

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

documents = text_splitter.create_documents(movie_description)

embeddings = OpenAIEmbeddings()

vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})



/var/folders/dr/8nqv5f4557v65lymsr24wbk00000gn/T/ipykernel_19232/3589715287.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [7]:
# Importing ChatOpenAI from LangChain to interact with OpenAI's language models, such as GPT, for generating responses.
from langchain_openai import ChatOpenAI

# Importing ChatPromptTemplate to create structured prompts for the chatbot, ensuring consistent interactions with the AI model.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Importing OpenAIEmbeddings to convert text data into numerical vector representations for similarity search and retrieval.
from langchain_openai import OpenAIEmbeddings

# Importing ChatPromptTemplate again (duplicate import, should be removed to avoid redundancy).
from langchain_core.prompts import ChatPromptTemplate

# Importing create_stuff_documents_chain to combine and process retrieved documents for meaningful AI-generated responses.
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Importing create_retrieval_chain to build a chain that retrieves relevant documents from a vector store and generates AI responses.
from langchain_classic.chains import create_retrieval_chain

# Importing StrOutputParser from LangChain to parse the output
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever


from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain




In [8]:
# Create the llm model
#llm = ChatOpenAI(api_key=os.environ["OPENAI_API_KEY"], model = 'gpt-5.4-nano')

llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model="openrouter/free", # Automatically dynamically routes to an open free model
    default_headers={
        "HTTP-Referer": "https://localhost:3000", # Optional, for OpenRouter analytics
        "X-Title": "My LangChain App",            # Optional, for OpenRouter rankings
    }
)


# Importing the output parser to process and format the model's response into a readable string format.
output_parser = StrOutputParser()


In [9]:
# Create the prompt template

# Creating a prompt template that instructs the AI to act as a movie information service agent.
# The prompt takes two parameters:
#   1. {context} - Relevant information retrieved from the document store.
#   2. {input} - The user's question.
# The model is instructed to base its answer solely on the provided context.

contextualize_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder("chat_history"), # Inject message array history dynamically
    ("human", "{input}"),
])

# Create history-aware search optimizer
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, prompt
)


In [10]:
# ============================================================
#  GROUNDED DOCUMENT ANSWERING CHAIN
# ============================================================
# This strict prompt forces the model to only look inside the context documents.
qa_system_prompt = (
    "You are an expert research assistant. Answer the user's question "
    "Use clear, conversational markdown and bullet points where appropriate."
    "using ONLY the provided retrieved context. If the answer cannot be found "
    "in the context documents, explicitly state: 'I am sorry, but that information is not in the context document.' Do not use external facts.\n\n"
    "Context:\n{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])


In [11]:

# Build combining stack and finalized RAG orchestration
document_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, document_answer_chain)


In [12]:

chat_history = [] # Acts as our short-term context memory repository
chat_history.append("Answer query as accurately as possible.")




In [13]:
import requests
import json
from langchain_core.load.dump import dumps
from langchain_core.messages import messages_to_dict, messages_from_dict
# First API call with reasoning

# Preserve the assistant message with reasoning_details
def preserve_reasoning_details(query: str,history: list = chat_history):
    messages = [
      {"role": "user", "content": f"{query}"},
      {"role": "assistant", "content": history[-1]},
       {"role": "user", "content": "Are you sure? Think carefully."},
    ]
    return messages
  
def query_with_history(query,history: list = chat_history):
    messages = preserve_reasoning_details(query,history)
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization":  f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        data=json.dumps({
            "model": "openrouter/free",
            "messages": messages,
            "reasoning": {"enabled": True}
        })
    )

    response = response.json()
    query_answer =  response['choices'][0]['message']
    final_response = str(query_answer['content'])
    temp_history = final_response.split('\n')[-1].split('\n')[-1]
    print(f"Temp history: {temp_history}")
    chat_history.append(temp_history)
    tokens_used = response['usage']['total_tokens']
    return final_response.replace('\n', ' ')



In [14]:
# Optional: Test the functionality using a Gradio UI (intermediate check)
import gradio as gr
from huggingface_hub import HfApi, whoami





def query_movie_chatbot(query,chat_history: list = chat_history) -> str:
    result = conversational_rag_chain.invoke({"input": query, "chat_history": chat_history})['answer']
    flag = ["I am sorry, but that information is not in the context document"]
    if any(f in result for f in flag):
        result = query_with_history(query)
    chat_history.append(result.split('\n')[-1])
    return result


chatbot_interface = gr.Interface(
    fn=query_movie_chatbot,
    inputs=gr.Textbox(label="Enter your query here"),
    outputs=gr.Textbox(label="Response"),
    title="Movie Chatbot",
    description="Ask me about movies!",
    theme="Glass",  # Optional: Choose a theme for the interface
    flagging_mode="never" 
)
chatbot_interface.launch(server_name="0.0.0.0", server_port=7860) 

# 1. Title and Description (Replaces 'title' and 'description')
"""st.title("Movie Chatbot")
st.caption("Ask me about movies!")

# 2. Input Component (Replaces gr.Textbox input)
user_query = st.text_input(label="Enter your query here", value="")

# 3. Form Submission Handling (Triggers 'fn=query_movie_chatbot')
if st.button("Submit", type="primary"):
    if user_query.strip():
        with st.spinner("Processing..."):
            # Call your existing function
            response = query_movie_chatbot(user_query)
        
        # 4. Output Component (Replaces gr.Textbox output)
        st.text_area(label="Response", value=response, height=150)
    else:
        st.warning("Please enter a query first.") """




/Users/uchegodfrey/Downloads/Ik_assignments/movie_chat_bot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/uchegodfrey/Downloads/Ik_assignments/movie_chat_bot/.venv/lib/python3.13/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


'st.title("Movie Chatbot")\nst.caption("Ask me about movies!")\n\n# 2. Input Component (Replaces gr.Textbox input)\nuser_query = st.text_input(label="Enter your query here", value="")\n\n# 3. Form Submission Handling (Triggers \'fn=query_movie_chatbot\')\nif st.button("Submit", type="primary"):\n    if user_query.strip():\n        with st.spinner("Processing..."):\n            # Call your existing function\n            response = query_movie_chatbot(user_query)\n\n        # 4. Output Component (Replaces gr.Textbox output)\n        st.text_area(label="Response", value=response, height=150)\n    else:\n        st.warning("Please enter a query first.") '

/Users/uchegodfrey/Downloads/Ik_assignments/movie_chat_bot/.venv/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Temp history: If you are looking for a specific film or era, let me know—but rest assured, Samuel L. Jackson is one of the highest-grossing actors of all time, and the list above represents only a curated highlight of his verified, major movie appearances.


# Test the performance of your bot with various test cases and refine your code!
